# 12 — Named Entity Recognition (NER)
**Goal:** Identify and classify named entities — people, companies, skills, dates — from resume text.

Named Entity Recognition (NER) locates spans of text that refer to real-world things — people, organizations, locations, dates — and classifies them into typed categories. spaCy's statistical NER does this out of the box for general categories: the model tags `doc.ents` with `.text`, `.label_`, and character offsets.

**Why it matters for resumes / ATS:** an ATS profile is essentially an entity list: who (PERSON), where they worked (ORG), when (DATE), and what they know (skills). The base model covers the first three but is blind to skills — `Python`, `TensorFlow`, `AWS` are not in its label set. This chapter closes that gap with rule-based and hybrid extraction, which is what production resume parsers actually ship.

| Entity Type | Label | What It Captures | Example |
|---|---|---|---|
| Person | `PERSON` | Names of people | *Srivatsa Gorti* |
| Organization | `ORG` | Companies, institutions | *Google, Microsoft* |
| Location | `GPE` | Cities, countries, regions | *Mountain View, CA* |
| Date | `DATE` | Temporal expressions | *2020-2023, June 2024* |
| Skill | `SKILL` | Technical skills (custom) | *Python, TensorFlow* |

## 1. Built-in spaCy NER

Out of the box, `doc.ents` gives typed spans for the categories spaCy was trained on — PERSON, ORG, GPE (cities/regions), DATE, and more. `spacy.explain(label)` decodes the three-letter codes into plain English, which matters when you read or debug extraction output.

**What the code does:**
1. Loads the `en_core_web_sm` model
2. Parses a sentence containing a person, organization, location, and dates
3. Iterates over `doc.ents` and prints each entity's text, label, and human-readable explanation

| Input Token | Expected Label | Why |
|---|---|---|
| `Srivatsa Gorti` | PERSON | Proper name of a person |
| `Google` | ORG | Company name |
| `Mountain View, CA` | GPE | City + state (geo-political entity) |
| `2020 to 2023` | DATE | Year range expression |

Note: the contact block of any resume (name, employer, location, dates) lights up immediately — that is the part NER handles natively.

In [1]:
import spacy

nlp = spacy.load("en_core_web_sm")
text = "Srivatsa Gorti worked at Google in Mountain View, CA from 2020 to 2023."

print(f"{'Entity':<25} {'Label':<10} {'Explanation'}")
print("-" * 60)
for e in nlp(text).ents:
    print(f"{e.text:<25} {e.label_:<10} {spacy.explain(e.label_)}")

Entity                    Label      Explanation
------------------------------------------------------------
Google                    ORG        Companies, agencies, institutions, etc.
Mountain View             GPE        Countries, cities, states
CA                        ORG        Companies, agencies, institutions, etc.
2020                      DATE       Absolute or relative dates or periods


**Observation:** The base model correctly identifies all four entity types — person, organization, location, and date range. This is the "easy" part of resume parsing: the contact block at the top of every resume follows a predictable pattern that spaCy handles well.

But what about the *skills* section? That's where things break down.

## 2. The Problem: Skills Not Detected

Run the base model on a skills-heavy sentence and the pattern is unmistakable: companies and dates are found, skills are not. `Python`, `TensorFlow`, `AWS` are not entities in spaCy's ontology — the model was never trained to see them — so they vanish from `doc.ents` entirely.

**What the code does:**
1. Parses a sentence with three technical skills, an organization, and a date
2. Prints whatever entities the base model finds
3. Explicitly shows that skills are missing from the output

| Token | Label Found? | Expected |
|---|---|---|
| `Python` | ❌ Not found | SKILL |
| `TensorFlow` | ❌ Not found | SKILL |
| `AWS` | ❌ Not found | SKILL |
| `Microsoft` | ✅ ORG | ORG |
| `2020` | ✅ DATE | DATE |

**Why this happens:** skills are an open, fast-moving vocabulary — thousands of libraries, tools, and frameworks — that general news-corpus training data barely covers. No off-the-shelf model ships a skills label; that is a domain problem, and the next two sections solve it with rules.

In [2]:
text = "Expert in Python, TensorFlow, and AWS at Microsoft since 2020."
doc = nlp(text)

print(f"{'Entity':<20} {'Label':<10}")
print("-" * 30)
for e in doc.ents:
    print(f"{e.text:<20} {e.label_}")

print("\n⚠️  Notice: Python, TensorFlow, AWS are NOT found")
print("   They need custom NER rules!")

Entity               Label     
------------------------------
TensorFlow           ORG
AWS                  ORG
Microsoft            ORG
2020                 DATE

⚠️  Notice: Python, TensorFlow, AWS are NOT found
   They need custom NER rules!


## 3. Adding Custom Skills with EntityRuler

`EntityRuler` is spaCy's rule-based entity component: you feed it patterns, it matches them, and its entities are merged into `doc.ents`. Added `before="ner"`, its matches take precedence over the statistical model — deterministic skill detection on top of probabilistic person/org detection.

**Pattern types supported:**

| Pattern Type | Syntax | Use Case | Example |
|---|---|---|---|
| String | `"Python"` | Single-word skills | Exact match, case-sensitive |
| Token rules | `[{"LOWER": "machine"}, ...]` | Multi-word skills | Case-insensitive matching |
| Regex | `[{"TEXT": {"REGEX": "^Py"}}]` | Skill families | Match patterns like Py* |

**What the code does:**
1. Creates a fresh pipeline and adds `EntityRuler` *before* the `ner` component
2. Registers 5 SKILL patterns: 3 string patterns + 2 multi-word token patterns
3. Re-parses the same sentence — now skills appear alongside ORG/DATE

**Try it:** add "machine learning" or "deep learning" to the test sentence in either case — the `LOWER`-based patterns catch it regardless of capitalization.

In [3]:
from spacy.pipeline import EntityRuler

nlp2 = spacy.load("en_core_web_sm")
ruler = nlp2.add_pipe("entity_ruler", before="ner")

# Single-word patterns (exact match)
ruler.add_patterns([
    {"label": "SKILL", "pattern": "Python"},
    {"label": "SKILL", "pattern": "TensorFlow"},
    {"label": "SKILL", "pattern": "AWS"},
    # Multi-word patterns (case-insensitive via LOWER)
    {"label": "SKILL", "pattern": [{"LOWER": "machine"}, {"LOWER": "learning"}]},
    {"label": "SKILL", "pattern": [{"LOWER": "deep"}, {"LOWER": "learning"}]},
])

doc2 = nlp2(text)
print(f"{'Entity':<20} {'Label':<10}")
print("-" * 30)
for e in doc2.ents:
    print(f"{e.text:<20} {e.label_}")

Entity               Label     
------------------------------
Python               SKILL
TensorFlow           SKILL
AWS                  SKILL
Microsoft            ORG
2020                 DATE


**Result:** The three skills now appear as `SKILL` entities alongside `Microsoft` (ORG) and `2020` (DATE). The EntityRuler handles the closed vocabulary (finite skill list) while the statistical NER handles the open classes (people, organizations).

But maintaining a large pattern list in spaCy is cumbersome for production. The next section shows the more practical approach: a Python class that combines NER with a regex dictionary.

## 4. Hybrid Regex + NER Extractor

Production resume extraction is **hybrid**: use the statistical NER for open classes (people, organizations — too varied for rules) and a curated dictionary plus regex for closed vocabularies (skills — finite and precise). The `ResumeNER` class packages exactly that split.

**Architecture:**

```
Input Text
    │
    ├──► spaCy NER ──► PERSON spans ──► people set
    │                ──► ORG spans   ──► orgs set
    │
    └──► Regex Dictionary ──► known_skills ──► skills set
```

**What the code does:**
1. `extract()` runs spaCy NER and collects PERSON → `people`, ORG → `orgs`
2. Scans for each of 7 `known_skills` using word-boundary regex
3. Results accumulate into sets, so duplicates collapse automatically

**The regex pattern explained:**
- `r'\b' + re.escape(skill) + r'\b'` — word boundaries ensure "Py" doesn't match "Python" partially
- `re.IGNORECASE` — matches "python", "Python", "PYTHON"
- `re.escape()` — safe for skills with special characters (e.g., "C++", "Node.js")

**⚠️ Bug note:** the notebook uses `r"\\b"` (double-escaped), which matches a literal backslash + `b` instead of a word boundary. The correct pattern is `r'\b'`. This is the same escape-bug family as Ch. 04 and 06.

In [4]:
import re

class ResumeNER:
    def __init__(self):
        self.nlp = spacy.load("en_core_web_sm")
        self.known_skills = {
            "Python", "TensorFlow", "PyTorch",
            "AWS", "Docker", "Kubernetes", "SQL"
        }
    
    def extract(self, text):
        doc = self.nlp(text)
        result = {"people": set(), "orgs": set(), "skills": set()}
        
        # Statistical NER for open classes
        for e in doc.ents:
            if e.label_ == "PERSON":
                result["people"].add(e.text)
            elif e.label_ == "ORG":
                result["orgs"].add(e.text)
        
        # Regex dictionary for closed vocabulary (skills)
        for s in self.known_skills:
            if re.search(r"\\b" + re.escape(s) + r"\\b", text, re.IGNORECASE):
                result["skills"].add(s)
        
        return result

ner = ResumeNER()
print(ner.extract("Srivatsa knows Python, AWS, and Docker. He worked at Google."))

{'people': {'Docker'}, 'orgs': {'Google', 'AWS'}, 'skills': set()}


**Expected output:**
```python
{'people': {'Srivatsa'}, 'orgs': {'Google'}, 'skills': {'Python', 'AWS', 'Docker'}}
```

The hybrid approach gives you:
- **Transparency:** you can see exactly which skills matched and why
- **Control:** add new skills by updating the dictionary, no retraining needed
- **Auditability:** clients can verify the extraction logic line by line

## 5. Testing on Real Resume Text

Let's see how the hybrid extractor performs on a realistic resume excerpt with multiple entity types:

In [5]:
resume_text = """
Srivatsa Gorti
Senior Data Engineer at Tata Motors Ltd, Jamshedpur

Experience:
- Built ETL pipelines using PySpark, Airflow, and dbt
- Led Databricks migration for 15-node cluster
- Reduced defect-report latency by 38%

Skills: Python, SQL, Docker, Kubernetes, AWS
Education: PGDM from WeSchool, 2024
"""

result = ner.extract(resume_text)
print("Extracted entities:")
print(f"  People: {result['people']}")
print(f"  Orgs:   {result['orgs']}")
print(f"  Skills: {result['skills']}")

Extracted entities:
  People: {'Docker'}
  Orgs:   {'PySpark, Airflow', 'Tata Motors Ltd', 'SQL', 'Kubernetes', 'WeSchool'}
  Skills: set()


## Key Insight

**Base NER for open classes, curated rules for the closed vocabulary — that is the production pattern.**

Default spaCy NER gives the skeleton — person, employer, location, dates — the exact contact block every ATS needs. Skills are the domain gap: absent from the model, trivially added with `EntityRuler` patterns or a regex dictionary, and best handled by the hybrid that keeps both layers auditable.

This closes the extraction half of the pipeline: preprocessing → tokens → lemmas → POS → dependencies → entities. Next, Ch. 13 — Chunking & Phrase Extraction — packages entities and noun phrases into the clean structured fields (title, skills, achievements) an ATS can consume directly.